# Scan degradation lab

Compare a clean score with reproducible photocopy simulations. Start with **Run All**:
exports are disabled by default. Then edit the settings and rerun from **Generate previews**.
Effects come from Augraphy: `BadPhotoCopy`, plus an optional `Geometric` → `PageBorder` → `BadPhotoCopy` → `Geometric` library-scan recipe. The notebook calls the same `preview`, `bad_photocopy`, and `export_preview` code as the CLI.

From the repository root, start with `uv run --group notebooks jupyter lab experiments/scan_degradation/scan_degradation.ipynb`.
Select the project's Python kernel. Poppler (`pdftoppm`) must be installed.
The example PDF lives locally in `data/reference_parts`; substitute another PDF if needed.


In [ ]:
from pathlib import Path
from copy import deepcopy
import json
import math
import shutil
import sys

# Works when Jupyter starts at the repository root or inside this experiment.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "experiments/scan_degradation/generate.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the MuSViT repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if shutil.which("pdftoppm") is None:
    raise RuntimeError("Install Poppler first (on macOS: brew install poppler).")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Markdown, display
from experiments.scan_degradation.generate import LIBRARY_SCAN, PROFILES, preview, export_preview

print(f"Python: {sys.executable}")
print(f"Repository: {ROOT}")


## Settings

Page numbers start at 1. Light-copy uses `SEED`; library-scan uses `SEED + 3`.
The library-scan seed stays fixed when other profiles are removed. Keep the seed fixed when comparing changes to one profile. Raster DPI
controls the source resolution, not an internal photocopier downsampling stage.

`COPY_PROFILE` is passed directly to **Augraphy `BadPhotoCopy`**; it is not our
own effect model. `noise_type` selects Gaussian (2), blobs (1), Perlin (3), Worley
(4), or rectangular (5) noise. `noise_side="none"` distributes noise across the
page; choose a side for a localized effect. `noise_iteration` controls mask
iterations, `noise_size` mask scale, and `noise_value` intensity (lower is darker).
`noise_sparsity`, `noise_concentration`, `blur_noise`, `wave_pattern` and
`edge_effect` control the noise appearance. Keep `numba_jit=0` for the seeded
preview workflow. The wrapper always applies the effect (`force=True`).

See [BadPhotoCopy parameters and examples](https://augraphy.readthedocs.io/en/latest/doc/source/augmentations/badphotocopy.html)
and [How Augraphy Works](https://augraphy.readthedocs.io/en/latest/doc/source/how_augraphy_works.html).
Other noise types can take longer on full pages; start with the Gaussian presets.

Current visual review: library-scan looks good; light-copy is acceptable. The
previous worn-copy and custom profiles were rejected for unrealistic noise and
are no longer generated. These judgments are visual feedback, not OMR metrics.


In [ ]:
PDF = ROOT / "data/reference_parts/Monsters_Inc-Bb_Clarinet.pdf"
PAGE = 1
SEED = 42
RENDER_DPI = 300

# Edit this accepted light-copy profile to experiment with its parameters.
COPY_PROFILE = deepcopy(PROFILES["light-copy"])

# These groups are direct kwargs for Geometric, PageBorder and BadPhotoCopy.
# Padding uses (left, right, top, bottom); floats are relative amounts.
SCAN_PROFILE = deepcopy(LIBRARY_SCAN)
SCAN_PROFILE["scanner_edge"]["padding"] = (0.002, 0.045, 0.003, 0.003)
SCAN_PROFILE["page_border"]["page_border_width_height"] = (0.035, 0.008)
SCAN_PROFILE["photocopy"]["noise_side"] = "right"
SCAN_PROFILE["framing"]["padding"] = (0.015, 0.025, 0.01, 0.055)
SCAN_PROFILE["framing"]["rotate_range"] = (-1, -1)

# Crop boxes are fractions of page width/height: left, top, right, bottom.
# Adjust these for a different PDF; each box is relative to its variant's canvas.
PASSAGE_BOX = (0.035, 0.085, 0.97, 0.30)
DETAIL_BOX = (0.08, 0.10, 0.38, 0.17)


## Generate previews

This rerasterizes the selected PDF page and computes all variants in memory.
Only the rasterizer's temporary files are written, then removed. It does not
change the reference PDF or existing exported runs.


In [ ]:
profiles = {"light-copy": deepcopy(COPY_PROFILE)}
source, variants, manifest = preview(
    PDF, page=PAGE, seed=SEED, profiles=profiles, render_dpi=RENDER_DPI, scan_profile=SCAN_PROFILE,
)
images = {"Clean reference": source, **variants}
print(f"Page {manifest['page']}: {source.shape[1]} × {source.shape[0]} pixels")
print(f"Source SHA-256: {manifest['source_sha256']}")
print(json.dumps({name: {"seed": info["seed"], **info["parameters"]}
                  for name, info in manifest["variants"].items()}, indent=2))


In [ ]:
def show_comparison(images, box=None, columns=3, max_width=1100):
    """Compare relative viewport crops; geometric variants are not registered."""
    if box is not None:
        left, top, right, bottom = box
        if not (0 <= left < right <= 1 and 0 <= top < bottom <= 1):
            raise ValueError("Crop fractions must satisfy 0 <= left < right <= 1 and 0 <= top < bottom <= 1.")
    crops, boxes = {}, {}
    for title, array in images.items():
        height, width = array.shape
        bounds = (0, 0, width, height) if box is None else (
            int(left * width), int(top * height), int(right * width), int(bottom * height)
        )
        if min(bounds[2] - bounds[0], bounds[3] - bounds[1]) < 1:
            raise ValueError("Choose a crop at least one pixel wide and high.")
        boxes[title] = bounds
        crops[title] = Image.fromarray(array).crop(bounds)
    first = next(iter(crops.values()))
    rows = math.ceil(len(images) / columns)
    row_height = max(2.3, min(8, 6 * first.height / first.width + 0.7))
    fig, axes = plt.subplots(rows, columns, figsize=(7 * columns, row_height * rows), squeeze=False)
    for ax, (title, crop) in zip(axes.flat, crops.items()):
        if crop.width > max_width:
            crop = crop.resize((max_width, max(1, round(crop.height * max_width / crop.width))), Image.Resampling.LANCZOS)
        ax.imshow(crop, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        ax.set_title(title, loc="left")
        ax.axis("off")
    for ax in axes.flat[len(images):]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()
    return boxes

## Whole page: paper inside a scan frame

The user-supplied library example has a slightly rotated sheet, extra scanner
canvas, a strong irregular right edge and lighter degradation across the notation.
`library-scan` approximates those separate effects with Augraphy's Geometric dark scanner edge, PageBorder,
localized BadPhotoCopy, and Geometric white padding/rotation. It is a starting recipe,
not a fitted reproduction of that scan. Inspect the page boundary as well as notes.

Adjust `SCAN_PROFILE` above; all groups are direct library parameters:
[PageBorder](https://augraphy.readthedocs.io/en/latest/doc/source/augmentations/pageborder.html),
[BadPhotoCopy](https://augraphy.readthedocs.io/en/latest/doc/source/augmentations/badphotocopy.html),
[Geometric](https://augraphy.readthedocs.io/en/latest/doc/source/augmentations/geometric.html).
Augraphy 8.2.6's Geometric rotation samples integer degrees; the initial value is
fixed at -1 degree. This recipe expands the output canvas instead of cropping music.

Displays are resized for viewing; arrays and exported PNGs keep full resolution.

In [ ]:
show_comparison(images, columns=3, max_width=700);


## Passage viewports

Compare staff lines, beam joins, chord text and multibar rests. BadPhotoCopy-only
variants retain the source coordinates. **Library-scan changes geometry and canvas
size**: its crop covers the same relative area of its own canvas, not an aligned
source-pixel region. Adjust the crop if needed; do not reuse source boxes as labels.

In [ ]:
passage_pixels = show_comparison(images, PASSAGE_BOX)
print("Passage viewport boxes in each image:", passage_pixels)


## Detail inspection

Inspect dots, accidentals, ties and thin strokes. Move `DETAIL_BOX` to a relevant
region in your own PDF and rerun this cell. A plausible photocopy effect can still
make a notation label ambiguous; record observations before using it in training.


In [ ]:
detail_pixels = show_comparison(images, DETAIL_BOX)
print("Detail viewport boxes in each image:", detail_pixels)


## Review notes

Edit this Markdown cell with what you observe. Keep the page, seeds and settings
above with your observations.

- Dots, accidentals and short rests:
- Staff lines, ties and beams:
- Text and multibar rest counts:
- Noise obscuring notation:
- Paper boundary, scanner margins and rotation:
- Profiles worth comparing against actual scans:

The profiles are exploratory simulations, not measured photocopier models.
No OMR accuracy has been evaluated here.


## Export the inspected run

Set `EXPORT = True` **only when ready**, choose a new directory, and run the next
cell. It saves the arrays and settings from the last preview generation, so later
edits to the settings cell cannot silently change the saved manifest. Existing
directories are rejected. Set the flag back to `False` after exporting.

The run contains `clean.png`, one PNG per variant, `comparison.png` (the CLI's
opening-passage contact sheet), `manifest.json`, and `inspection.json` with your
crop boxes and review note. Crop boxes belong to each variant's own viewport.
Library-scan changes geometry; no source-to-output coordinate mapping is exported.
Source spatial labels must be transformed separately before spatial training.
The manifest records the Augraphy version and exact effect parameters.


In [ ]:
EXPORT = False
EXPORT_DIR = ROOT / "experiments/scan_degradation/outputs/reviewed_profiles_run_001"
REVIEW_NOTE = ""  # Add observations to accompany this exported run.

if EXPORT:
    destination = export_preview(source, variants, manifest, EXPORT_DIR)
    inspection = {
        "passage_boxes_pixels": passage_pixels,
        "detail_boxes_pixels": detail_pixels,
        "note": REVIEW_NOTE,
    }
    (destination / "inspection.json").write_text(json.dumps(inspection, indent=2) + "\n")
    print(f"Saved inspected run: {destination}")
else:
    print("Preview only; nothing exported. Set EXPORT = True to save a new run.")
